In [1]:

from urllib.parse import urljoin
from typing import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
from IPython.display import Image, display
import gradio as gr
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
import os
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict

import sqlite3
import pandas as pd



/Users/caineosborne/Projects2026/demographics-agent/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
db_path = "../Data_Files/WPP2024_GEN_F01_DEMOGRAPHIC_INDICATORS_COMPACT.sqlite"

In [3]:
with sqlite3.connect(db_path) as conn:
    # estimates = pd.read_sql_query("SELECT * FROM estimates", conn)
    medium_variant = pd.read_sql_query("SELECT * FROM medium_variant", conn)
medium_variant.columns
# medium_variant.head()
# medium_variant[medium_variant['Country'] == 'Viet Nam']

Index(['Index', 'Variant', 'Country', 'Notes', 'Location code',
       'ISO3 Alpha-code', 'ISO2 Alpha-code', 'SDMX code**', 'Type',
       'Parent code', 'Year', 'Population 1 Jan', 'Population 1 Jul',
       'Male Population, as of 1 July (thousands)',
       'Female Population, as of 1 July (thousands)',
       'Population Density, as of 1 July (persons per square km)',
       'Population Sex Ratio, as of 1 July (males per 100 females)',
       'Median Age, as of 1 July (years)', 'Natural Change',
       'Rate of Natural Change (per 1,000 population)',
       'Population Change (thousands)', 'Population Growth Rate (percentage)',
       'Population Annual Doubling Time (years)', 'Total Births',
       'Births by women aged 15 to 19 (thousands)',
       'Crude Birth Rate (births per 1,000 population)',
       'Total Fertility Rate (live births per woman)',
       'Net Reproduction Rate (surviving daughters per woman)',
       'Mean Age Childbearing (years)',
       'Sex Ratio at Birth

In [4]:
def get_connection():
    return sqlite3.connect(db_path)

def run_query(sql, params=()):
    with sqlite3.connect(db_path) as conn:
        conn.row_factory = sqlite3.Row
        rows = conn.execute(sql, params).fetchall()
        return [dict(row) for row in rows]

In [ ]:
@tool
def get_population_forecast(country: str, year: int):
    "retreive the estimated population for a given country and year from the UN data"
    sql = """
        SELECT
            Country, 
            Year,
            "Population 1 Jan"
        FROM medium_variant
        WHERE Country = ?
          AND Year = ?
    """
    print("Getting population for", country, year)
    return run_query(sql, (country, year))

In [ ]:
@tool
def get_list_of_countries() -> list[str]:
    """List the exact country and area names available in the UN dataset.

    Use this when you are unsure how a location is named in the database.
    """
    sql = """
        SELECT DISTINCT country AS Country
        FROM medium_variant
        ORDER BY country
    """

    print("Getting list of countries")

    return [row["Country"] for row in run_query(sql)]

In [ ]:
tools = [get_population_forecast,get_list_of_countries]

In [ ]:
memory = MemorySaver()

load_dotenv(override=True)
api_key=os.getenv("OPENROUTER_API_KEY")

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]


graph_builder = StateGraph(State)


llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
    model="qwen/qwen3.7-flash"
)


llm_with_tools = llm.bind_tools(tools)


def chatbot(state: State):
    return {
        "messages": [
            llm_with_tools.invoke(state["messages"])
        ]
    }


graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", ToolNode(tools=tools))


graph_builder.add_edge(START, "chatbot")

graph_builder.add_conditional_edges(
    "chatbot",
    tools_condition
)

graph_builder.add_edge("tools", "chatbot")


graph = graph_builder.compile(checkpointer=memory)

In [ ]:
config = {"configurable": {"thread_id": "1"}}

def chat(user_input: str, history):
    result = graph.invoke({"messages": [{"role": "user", "content": user_input}]}, config=config)
    return result["messages"][-1].content


gr.ChatInterface(chat).launch()

In [ ]:
response = llm_with_tools.invoke(
    "What is Australia's estimated population in 2030?"
)

print(response.tool_calls)